In [2]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# 1. 데이터 로드 및 전처리 (데이터셋 파일이 로컬에 있다고 가정)
# 관리를 위해 주요 3개 테이블만 사용합니다.
orders = pd.read_csv('olist/olist_orders_dataset.csv')
items = pd.read_csv('olist/olist_order_items_dataset.csv')
customers = pd.read_csv('olist/olist_customers_dataset.csv')

# 데이터 병합: 고객 정보 + 주문 정보 + 상품 가격 정보
df = orders.merge(customers, on='customer_id').merge(items, on='order_id')
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

# 2. [Funnel & Feature Engineering] 마케팅 피처 생성
# 분석 기준일 설정 (데이터상 마지막 날짜)
snapshot_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

# 고객별 RFM 및 행동 지표 산출
user_features = df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days, # Recency
    'order_id': 'nunique', # Frequency
    'price': 'sum', # Monetary (LTV의 기초)
    'freight_value': 'mean' # 배송비 부담 정도 (이탈 원인 가설)
}).reset_index()

user_features.columns = ['user_id', 'recency', 'frequency', 'monetary', 'avg_freight']

# 3. [ML Target] 이탈(Churn) 정의
# 최근 6개월(180일)간 구매가 없으면 이탈(1)로 정의
user_features['is_churned'] = (user_features['recency'] > 180).astype(int)

# 4. [Machine Learning] 이탈 예측 모델링
X = user_features[['frequency', 'monetary', 'avg_freight']] # 예측 변수
y = user_features['is_churned'] # 타겟 변수

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# XGBoost 모델 학습
model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
model.fit(X_train, y_train)

# 예측 및 성능 확인
churn_probs = model.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, churn_probs)

# 5. [Growth Metrics] 비즈니스 임팩트 산출
print("--- [Olist 실전 데이터 분석 리포트] ---")
print(f"1. 분석 대상 총 고객 수: {len(user_features):,}")
print(f"2. 전체 평균 인당 매출 (ARPU): {user_features['monetary'].mean():.2f} BRL")
print(f"3. ML 모델 이탈 예측 성능 (AUC): {auc_score:.4f}")

# 가상의 마케팅 시나리오 적용 (LTV/CAC)
# Olist의 마케팅 데이터를 참고하여 평균 CAC를 30 BRL로 가정 시
estimated_cac = 30 
avg_ltv = user_features['monetary'].mean()
ltv_cac_ratio = avg_ltv / estimated_cac

print(f"4. 예상 비즈니스 건강도 (LTV/CAC): {ltv_cac_ratio:.2f}x")

--- [Olist 실전 데이터 분석 리포트] ---
1. 분석 대상 총 고객 수: 95,420
2. 전체 평균 인당 매출 (ARPU): 142.44 BRL
3. ML 모델 이탈 예측 성능 (AUC): 0.9130
4. 예상 비즈니스 건강도 (LTV/CAC): 4.75x


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [04:28:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
